# iiot3w — 02. Second batch on Colab: baselines, multiclass, rotation (T29, T27, T26)

Same environment and procedure as `01_run_all_colab.ipynb` (CPU, High-RAM, one thread
per seed, resumable per seed in Drive). Runs that need new model fits:

| Run | Partition | Selection | Arms | Labels | Withheld | Purpose |
|---|---|---|---|---|---|---|
| `run03_families` | wells | nested | rf, transformer, lgbm, cnn | binary | 1, 8, 9 | T29: model families (RQ4) |
| `run04_multiclass` | wells | nested | rf, transformer | multiclass | 1, 8, 9 | T27: empty-set rejection under well shift (RQ4) |
| `run05_rot<c>` | wells | inner | rf, transformer | binary | `c` | T26: rotation of the withheld type (RQ3) |

Rotation codes with at least 73 unseen test windows in every partition: 3, 4, 6, 8
(checked 2026-11-20). Each run is independent and resumable: run the whole notebook again in a new session
and the seeds already saved in Drive are skipped. Section 5 (post-hoc analyses of
batch 1) takes minutes and runs first in every session.

* **Prerequisite:** `iiot3w_code.zip` in `My Drive/pesquisa/iiot3w/` built from a commit
  that contains `iiot3w/temporal.py` and the `--arms/--label-mode/--withheld` options.

## 1. Configuration

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pesquisa/iiot3w"   # project folder in Google Drive
CODE_ZIP = f"{DRIVE_ROOT}/iiot3w_code.zip"              # repository folder code/, zipped
SEEDS = [0, 1, 2, 3, 4]
BIN = {"label_mode": "binary", "withheld": None}
RUNS = {   # run id -> run_all.py options
    "run03_families":   {"split_mode": "wells", "selection": "nested",
                         "arms": "rf,transformer,lgbm,cnn", **BIN},
    "run04_multiclass": {"split_mode": "wells", "selection": "nested",
                         "arms": "rf,transformer", "label_mode": "multiclass", "withheld": None},
    **{f"run05_rot{c}": {"split_mode": "wells", "selection": "inner",
                         "arms": "rf,transformer", "label_mode": "binary", "withheld": str(c)}
       for c in (3, 4, 6, 8)},
}
SELECT_ON_SEED = 0         # with inner selection, the other seeds reuse this seed's choice
N_PARALLEL = 2             # 2 CPUs on Colab; each seed uses 1 thread

## 2. Setup: Drive, code, dependencies, environment record

In [ ]:
import datetime, hashlib, json, os, shutil, subprocess, sys, time, zipfile
from google.colab import drive
drive.mount("/content/drive")


def utc_now():
    """Current UTC time (ISO 8601), for logs and provenance."""
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def sh(cmd, log=None, check=True):
    """Run a shell command, echo and optionally save its output."""
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = p.stdout + p.stderr
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        open(log, "a").write(f"$ {cmd}\n{out}\n")
    if check and p.returncode != 0:
        print(out[-3000:])
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return out


RUNS_DIR = f"{DRIVE_ROOT}/runs"
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
ENV, LOGS = f"{RUNS_DIR}/env/session_{SESSION}", f"{RUNS_DIR}/logs"
for d in (RUNS_DIR, ENV, LOGS):
    os.makedirs(d, exist_ok=True)

# ---- Code: unzip a fresh copy and record its hash ---------------------------
CODE = "/content/code"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall("/content")
assert os.path.exists(f"{CODE}/run_all.py"), "the zip must contain the folder code/"
h = hashlib.sha256()
for root, _, files in sorted(os.walk(f"{CODE}/iiot3w")):
    for f in sorted(files):
        if f.endswith(".py"):
            h.update(open(os.path.join(root, f), "rb").read())
CODE_HASH = h.hexdigest()[:16]
print("code hash (iiot3w/*.py):", CODE_HASH)

# ---- Dependencies: the pinned versions of requirements.txt, CPU-only torch ----
sh(f"{sys.executable} -m pip install -q torch==2.12.0 --index-url https://download.pytorch.org/whl/cpu",
   log=f"{ENV}/pip_install.log")
sh(f"{sys.executable} -m pip install -q -r {CODE}/requirements.txt", log=f"{ENV}/pip_install.log")

# ---- Environment record --------------------------------------------------------
open(f"{ENV}/pip_freeze.txt", "w").write(sh(f"{sys.executable} -m pip freeze"))
open(f"{ENV}/lscpu.txt", "w").write(sh("lscpu", check=False))
open(f"{ENV}/memory.txt", "w").write(sh("free -h", check=False))
N_CPU = os.cpu_count()
N_PAR = N_PARALLEL or max(1, N_CPU - 1)
json.dump({"session": SESSION, "start": utc_now(), "code_hash": CODE_HASH, "cpu_count": N_CPU,
           "parallel_seeds": N_PAR, "python": sys.version, "seeds": SEEDS, "runs": RUNS},
          open(f"{ENV}/session.json", "w"), indent=1)
print(f"CPUs: {N_CPU}; seeds in parallel: {N_PAR}")
print(sh("free -h", check=False))

## 3. Unit tests

In [ ]:
print(sh(f"cd {CODE} && {sys.executable} -m pytest tests -q -p no:cacheprovider", log=f"{LOGS}/tests.log")[-400:])

## 4. Data: 3W 2.0.0 at the pinned commit, audit and windowing

In [ ]:
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f"{CODE}/data/3W/dataset"):
    print(sh(f"cd {CODE} && bash scripts/fetch_3w.sh data/3W", log=f"{LOGS}/fetch_3w.log"))
sh(f"cd {CODE} && {sys.executable} run_all.py --dataset-dir data/3W/dataset --stages audit --out {WORK}/ref",
   log=f"{LOGS}/audit.log")

# The audit must be identical to the one of the released reference run.
new = json.load(open(f"{WORK}/ref/audit.json"))
ref = json.load(open(f"{CODE}/results/cpu_reference/audit.json"))
assert new == ref, "audit.json differs from results/cpu_reference/audit.json"
shutil.copy2(f"{WORK}/ref/audit.json", f"{RUNS_DIR}/audit.json")
print("audit identical to the reference run:", new["n_instances"], "instances,", new["n_windows"], "windows")

## 5. Post-hoc analyses of batch 1 with this code (no refitting)

Reruns the stages that do not fit models (asset, novelty, temporal) on the merged
results of batch 1 (`ref_colab`, `run01_wells`, `run02_wells_nested`), so that every
analysis of the article comes from the same code and environment. Writes into
`runs/<run>/posthoc/` in Drive; the batch-1 folders are not modified.


In [ ]:
BATCH1 = {"ref_colab": "windows", "run01_wells": "wells", "run02_wells_nested": "wells"}
for run, mode in BATCH1.items():
    src = f"{RUNS_DIR}/{run}/merged"
    if not os.path.exists(f"{src}/experiment.json"):
        print(f"{run}: merged results not found, skipped")
        continue
    P = f"{WORK}/posthoc/{run}"
    shutil.rmtree(P, ignore_errors=True)
    shutil.copytree(src, P)
    print(sh(f"cd {CODE} && {sys.executable} run_all.py --split-mode {mode} --out {P} "
             f"--provenance-dir {WORK}/ref --stages asset,novelty,temporal",
             log=f"{LOGS}/{run}/posthoc.log")[-1500:])
    dst = f"{RUNS_DIR}/{run}/posthoc"
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(P, dst, ignore=shutil.ignore_patterns("windows.npz"))
    print(f"saved {dst}")


## 6. Experiments: one process per (run, seed), one thread each

A shared queue runs up to `N_PARALLEL` seeds at once across the three runs. With inner
selection, seed 0 of a run must finish before its other seeds start, because they reuse
its hyperparameters (`--selection-from`). Seeds already saved in Drive are skipped.
Rough cost per seed on one thread: 10–20 min with inner selection, 40–60 min with nested.

In [ ]:
def run_flags(o):
    """Options of batch 2 (arms, label mode, withheld codes)."""
    f = ["--arms", o["arms"], "--label-mode", o["label_mode"]]
    return f + (["--withheld", o["withheld"]] if o.get("withheld") else [])


def seed_dir(run, s):
    return f"{RUNS_DIR}/{run}/s{s}"


def seed_done(run, s):
    return os.path.exists(f"{seed_dir(run, s)}/experiment.json")


def ready(run, s):
    """Inner selection: seeds other than SELECT_ON_SEED wait for it."""
    return RUNS[run]["selection"] == "nested" or s == SELECT_ON_SEED or seed_done(run, SELECT_ON_SEED)


def command(run, s, out):
    o = RUNS[run]
    cmd = [sys.executable, "run_all.py", "--split-mode", o["split_mode"], "--selection", o["selection"],
           "--stages", "experiment", "--seeds", str(s), "--threads", "1", "--out", out] + run_flags(o)
    if o["selection"] == "inner" and s != SELECT_ON_SEED:
        cmd += ["--selection-from", f"{seed_dir(run, SELECT_ON_SEED)}/experiment.json"]
    return cmd


# Nested seeds first (longest), then the selection seeds of the inner runs.
queue = sorted(((r, s) for r in RUNS for s in SEEDS if not seed_done(r, s)),
               key=lambda x: (RUNS[x[0]]["selection"] != "nested", x[1] != SELECT_ON_SEED, x))
print("to run:", queue)
running, t0 = {}, time.time()
while queue or running:
    for job in list(queue):
        if len(running) >= N_PAR:
            break
        if not ready(*job):
            continue
        queue.remove(job)
        run, s = job
        out = f"{WORK}/{run}/s{s}"
        os.makedirs(out, exist_ok=True)
        for f in ("windows.npz", "audit.json"):
            shutil.copy2(f"{WORK}/ref/{f}", out)
        running[job] = (subprocess.Popen(command(run, s, out), cwd=CODE, stdout=open(f"{out}.stdout", "w"),
                                         stderr=subprocess.STDOUT), out)
        print(f"[{utc_now()}] {run} seed {s} started")
    time.sleep(60)
    for job, (p, out) in list(running.items()):
        if p.poll() is None:
            continue
        del running[job]
        run, s = job
        os.makedirs(f"{LOGS}/{run}", exist_ok=True)
        shutil.copy2(f"{out}.stdout", f"{LOGS}/{run}/seed{s}.stdout")
        if p.returncode != 0:
            print(open(f"{out}.stdout").read()[-2000:])
            raise RuntimeError(f"{run} seed {s} failed ({p.returncode})")
        dst = seed_dir(run, s)   # copy to Drive: the seed is now safe from a disconnect
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(out, dst, ignore=shutil.ignore_patterns("windows.npz"))
        print(f"[{utc_now()}] {run} seed {s} done and saved ({(time.time() - t0) / 60:.0f} min elapsed)")
    if running:
        last = {f"{r}/s{s}": (open(f"{o}.stdout").read().strip().splitlines() or [""])[-1][:60]
                for (r, s), (_, o) in running.items()}
        print(f"[{utc_now()}] running:", last)
print("all seeds of all runs done")

## 7. Merge and analysis per run (analysis, robustness, asset, novelty, temporal, figures)

In [ ]:
for run, o in RUNS.items():
    M = f"{WORK}/merge/{run}"
    shutil.rmtree(M, ignore_errors=True)
    for s in SEEDS:   # bring every seed from Drive (also those of earlier sessions)
        shutil.copytree(seed_dir(run, s), f"{M}/s{s}")
    shutil.copy2(f"{WORK}/ref/windows.npz", f"{M}/s{SELECT_ON_SEED}/windows.npz")
    print(sh(f"cd {CODE} && {sys.executable} scripts/merge_seeds.py {M}", log=f"{LOGS}/{run}/merge.log"))
    stages = "analysis,robustness,asset,novelty,temporal,figures"
    flags = " ".join(run_flags(o))
    print(sh(f"cd {CODE} && {sys.executable} run_all.py --split-mode {o['split_mode']} --selection {o['selection']} "
             f"{flags} --out {M} --provenance-dir {WORK}/ref --stages {stages}",
             log=f"{LOGS}/{run}/analysis.log")[-2500:])
    # Merged results (without windows.npz) and a zip of the run, to Drive.
    dst = f"{RUNS_DIR}/{run}/merged"
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(M, dst, ignore=shutil.ignore_patterns("windows.npz", "s[0-9]*"))
    shutil.make_archive(f"{RUNS_DIR}/{run}/{run}", "zip", dst)
    print(f"saved {run}: {dst}")
json.dump({"end": utc_now(), "code_hash": CODE_HASH}, open(f"{ENV}/session_end.json", "w"))
print("batch 2 done")